In [1]:
import os
print(os.path.exists('/PHShome_actual/l/ll1009/softwares/scDRS'))
print(os.listdir('/PHShome/ll1009') if os.path.exists('/PHShome/ll1009') else "no PHShome")

False
['linke', 'scratch', '.cache', '.conda', '.config', '.ipynb_checkpoints', '.ipython', '.jupyter', '.local', '.ssh', '.subversion', '.vscode-server', 'R', 'lsf', '.wget-hsts', '.bash_logout', '.bash_profile', '.bashrc', '.emacs', '.forward', '.mysql_history', '.zshrc', '.Rhistory', '.sh_history', '.gitconfig', '.viminfo', 'softwares', '.python_history', '.lesshst', '.bash_history']


In [11]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy as sp
import scipy.sparse as sp_sparse
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from anndata import AnnData
import os
import time
from gprofiler import GProfiler

# scTRS tools
import scdrs.util as util
import scdrs.data_loader as dl
import scdrs.method as md

# autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
os.chdir('/PHShome/ll1009/linke/chip_gwas_rev/')

## Making covariate file for BL_hashing.h5ad

In [ ]:
DATA_FILE='Data/sc_annot_files/BL_hashing.h5ad'
OUT_PATH='Data/sc_annot_files_qc/BL_hashing_covs.tsv'

In [ ]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [ ]:
# Check what's in .X
print("X dtype:", adata_full.X.dtype)
print("X sample values:")
print(adata_full.X[:3, :5].toarray() if hasattr(adata_full.X, 'toarray') else adata_full.X[:3, :5])
print("X min/max:", adata_full.X.min(), adata_full.X.max())

# Check if raw exists
print("\nHas raw:", adata_full.raw is not None)

# Check layers
print("\nLayers:", list(adata_full.layers.keys()) if adata_full.layers else "none")

# Check uns keys for count matrices
print("\nuns keys:", list(adata_full.uns.keys()))

# Check obs columns in detail
print("\nobs dtypes:")
print(adata_full.obs.dtypes)

# Check what anno looks like
print("\nAnno values:")
print(adata_full.obs['anno'].value_counts().head(10))

# Check assignment — is it donor?
print("\nAssignment values:")
print(adata_full.obs['assignment'].value_counts().head(10))

In [ ]:
# Check if low-quality cells are already removed
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Any predicted doublets:", adata_full.obs['pred_dbl'].sum())

In [ ]:
# Filter: remove doublets AND any unconfidently assigned cells
clean_mask = (
    (adata_full.obs['pred_dbl'] == False) &
    (adata_full.obs['assignment'].isin(['MantonBL1', 'MantonBL2', 'MantonBL3',
                                        'MantonBL4', 'MantonBL5', 'MantonBL6',
                                        'MantonBL7', 'MantonBL8']))
)
adata_full = adata_full[clean_mask].copy()
print("Clean cells:", adata_full.shape[0])

# Covariate file — with donor dummies (appropriate for hashing data)
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

# Donor dummies — drop MantonBL1 as reference
donors = sorted(adata_full.obs['assignment'].unique())
print("Donors:", donors)
for donor in donors[1:]:
    df_cov['donor_%s' % donor] = (adata_full.obs['assignment'] == donor).astype(int)

    
print("\nCov shape:", df_cov.shape)
print(df_cov.head())

In [ ]:
### saving to csv files
df_cov.to_csv(OUT_PATH, sep="\t")

## Making covariate file for BL_standard_design.h5ad

In [ ]:
DATA_FILE='Data/sc_annot_files/BL_standard_design.h5ad'
OUT_PATH='Data/sc_annot_files_qc/BL_standard_design_covs.tsv'

In [ ]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [ ]:
adata_full

In [ ]:
# Check filtering status
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Predicted doublets:", adata_full.obs['pred_dbl'].sum())

# Check key columns
print("\nDonor value counts:")
print(adata_full.obs['Donor'].value_counts().head(15))

print("\nChannel value counts:")
print(adata_full.obs['Channel'].value_counts().head(15))

print("\nGroup value counts:")
print(adata_full.obs['Group'].value_counts())

print("\nAnno value counts:")
print(adata_full.obs['anno'].value_counts().head(15))

# Check X
print("\nX dtype:", adata_full.X.dtype)
print("X sample values:")
import numpy as np
sample = adata_full.X[:3, :5]
print(sample.toarray() if hasattr(sample, 'toarray') else sample)
print("X min/max:", adata_full.X.min(), adata_full.X.max())
print("Has raw:", adata_full.raw is not None)

In [ ]:
# Check if low-quality cells are already removed
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Any predicted doublets:", adata_full.obs['pred_dbl'].sum())

In [ ]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

channels = sorted(adata_full.obs['Channel'].unique())
print(f"N channels: {len(channels)}")
for ch in channels[1:]:   # drop first as reference
    df_cov['channel_%s' % ch] = (
        adata_full.obs['Channel'] == ch).astype(int)

In [ ]:
# How many cells per channel?
print(adata_full.obs['Channel'].value_counts().describe())

# Do channel names look like sample IDs or run IDs?
print(adata_full.obs['Channel'].value_counts().head(10))

### these are actual donors

In [ ]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

channels = sorted(adata_full.obs['Channel'].unique())
print(f"N channels: {len(channels)}")  # expect 64
for ch in channels[1:]:               # 63 dummy columns
    df_cov['channel_%s' % ch] = (
        adata_full.obs['Channel'] == ch).astype(int)

print("Shape:", df_cov.shape)         # expect (323269, 65)

In [ ]:
### saving to csv files
df_cov.to_csv(OUT_PATH, sep="\t")

## Making covariate file for BM_pooling_and_control.h5ad

In [4]:
DATA_FILE='Data/sc_annot_files/BM_pooling_and_control.h5ad'
OUT_PATH='Data/sc_annot_files_qc/BM_pooling_and_control_covs.tsv'

In [5]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [6]:
adata_full

AnnData object with n_obs × n_vars = 159708 × 26254
    obs: 'n_genes', 'n_counts', 'percent_mito', 'Channel', 'assignment', 'anno'
    var: 'featureid', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features'
    uns: 'PCs', 'genome', 'modality', 'pca', 'pca_features'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap', 'pca_harmony_knn_distances', 'pca_harmony_knn_indices'
    obsp: 'W_pca_harmony'

In [7]:
# Quality check
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())

# No pred_dbl column this time — check if it's somewhere else
print("obs columns:", adata_full.obs.columns.tolist())

# Channel structure
print("\nChannel value counts (describe):")
print(adata_full.obs['Channel'].value_counts().describe())
print("\nChannel top 10:")
print(adata_full.obs['Channel'].value_counts().head(10))

# Assignment — donor or demux classification?
print("\nAssignment unique values:", adata_full.obs['assignment'].nunique())
print(adata_full.obs['assignment'].value_counts().head(10))

# X check
sample = adata_full.X[:3, :5]
print("\nX sample:", sample.toarray() if hasattr(sample, 'toarray') else sample)
print("X min/max:", adata_full.X.min(), adata_full.X.max())

N cells: 159708
Min n_genes: 500
Max percent_mito: 9.998043435726863
obs columns: ['n_genes', 'n_counts', 'percent_mito', 'Channel', 'assignment', 'anno']

Channel value counts (describe):
count      24.000000
mean     6654.500000
std      2076.489219
min      3294.000000
25%      4306.000000
50%      7865.000000
75%      8233.250000
max      8902.000000
Name: count, dtype: float64

Channel top 10:
Channel
BMP3     8902
BMP16    8863
BMP1     8639
BMP13    8476
BMP2     8463
BMP14    8258
BMP10    8225
BMP7     8209
BMP6     8135
BMP12    8083
Name: count, dtype: int64

Assignment unique values: 9
assignment
          31359
MK.BM6    18508
MK.BM2    18242
MK.BM4    17442
MK.BM5    17056
MK.BM8    14949
MK.BM1    14097
MK.BM3    14093
MK.BM7    13962
Name: count, dtype: int64

X sample: [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
X min/max: 0 17375


In [8]:
# Check if low-quality cells are already removed
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())

N cells: 159708
Min n_genes: 500
Max percent_mito: 9.998043435726863


In [12]:
### need to remove the cells without a proper assignment (there were 31359 cells without actual assignment)
# Check blank assignment
print("Before filter:", adata_full.shape)
print("Blank assignments:", (adata_full.obs['assignment'] == '').sum())
print("NA assignments:", adata_full.obs['assignment'].isna().sum())

# The blank category
blank_mask = adata_full.obs['assignment'].isin(['', 'unassigned',
                                                'doublet', 'Doublet'])
print("Cells to remove:", blank_mask.sum())

# Filter to confidently assigned cells only
valid_donors = ['MK.BM1', 'MK.BM2', 'MK.BM3', 'MK.BM4',
                'MK.BM5', 'MK.BM6', 'MK.BM7', 'MK.BM8']
adata = adata_full[adata_full.obs['assignment'].isin(valid_donors)].copy()
print("After filter:", adata.shape)


#### converting float32 to 64
# Convert X to float32 so scDRS can normalize it
print("X dtype before:", adata.X.dtype)
if sp_sparse.issparse(adata.X):
    adata.X = adata.X.astype(np.float32)
else:
    adata.X = adata.X.astype(np.float32)
print("X dtype after:", adata.X.dtype)

# Save
adata.write("BM_pooling_and_control_filtered.h5ad")
print("Saved with float32 X")

Before filter: (159708, 26254)
Blank assignments: 31359
NA assignments: 0
Cells to remove: 31359
After filter: (128349, 26254)
X dtype before: uint32
X dtype after: float32
Saved with float32 X


In [13]:
### creating file
# Generate covariate file from filtered object
df_cov = pd.DataFrame(index=adata.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata.obs['n_genes']

# Channel dummies (15 columns, drop first as reference)
channels = sorted(adata.obs['Channel'].unique())
print(f"N channels: {len(channels)}")
print(f"Reference channel (dropped): {channels[0]}")
for ch in channels[1:]:
    df_cov['channel_%s' % ch] = (
        adata.obs['Channel'] == ch).astype(int)

print(f"\nFinal cov shape: {df_cov.shape}")
# Expect (~128K cells, 17 columns: const + n_genes + 15 channel dummies)
print(df_cov.head())

N channels: 16
Reference channel (dropped): BMP1

Final cov shape: (128349, 17)
                       const  n_genes  channel_BMP10  channel_BMP11  \
barcodekey                                                            
BMP1-AAACCTGAGAGGACGG      1     2801              0              0   
BMP1-AAACCTGAGATCGATA      1     1669              0              0   
BMP1-AAACCTGAGATGAGAG      1      726              0              0   
BMP1-AAACCTGAGGTGACCA      1     1634              0              0   
BMP1-AAACCTGCAATGCCAT      1      596              0              0   

                       channel_BMP12  channel_BMP13  channel_BMP14  \
barcodekey                                                           
BMP1-AAACCTGAGAGGACGG              0              0              0   
BMP1-AAACCTGAGATCGATA              0              0              0   
BMP1-AAACCTGAGATGAGAG              0              0              0   
BMP1-AAACCTGAGGTGACCA              0              0              0   
BM

In [ ]:
### saving to csv files
df_cov.to_csv(OUT_PATH, sep="\t")
adata.write("Data/sc_annot_files/BM_pooling_and_control_filtered.h5ad")
print("\nSaved bm_pooling.cov and bm_pooling_filtered.h5ad")

## Making covariate file for BM_standard_design.h5ad

In [ ]:
DATA_FILE='Data/sc_annot_files/BM_standard_design.h5ad'
OUT_PATH='Data/sc_annot_files_qc/BM_standard_design_covs.tsv'

In [ ]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [ ]:
adata_full

In [ ]:
# Check filtering status
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Predicted doublets:", adata_full.obs['pred_dbl'].sum())

# Check key columns
print("\nDonor value counts:")
print(adata_full.obs['Donor'].value_counts().head(15))

print("\nChannel value counts:")
print(adata_full.obs['Channel'].value_counts().head(15))

print("\nGroup value counts:")
print(adata_full.obs['Group'].value_counts())

print("\nAnno value counts:")
print(adata_full.obs['anno'].value_counts().head(15))

# Check X
print("\nX dtype:", adata_full.X.dtype)
print("X sample values:")
import numpy as np
sample = adata_full.X[:3, :5]
print(sample.toarray() if hasattr(sample, 'toarray') else sample)
print("X min/max:", adata_full.X.min(), adata_full.X.max())
print("Has raw:", adata_full.raw is not None)

In [ ]:
# Check if low-quality cells are already removed
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Any predicted doublets:", adata_full.obs['pred_dbl'].sum())

In [ ]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

channels = sorted(adata_full.obs['Channel'].unique())
print(f"N channels: {len(channels)}")
for ch in channels[1:]:   # drop first as reference
    df_cov['channel_%s' % ch] = (
        adata_full.obs['Channel'] == ch).astype(int)

In [ ]:
# How many cells per channel?
print(adata_full.obs['Channel'].value_counts().describe())

# Do channel names look like sample IDs or run IDs?
print(adata_full.obs['Channel'].value_counts().head(10))

### these are actual donors

In [ ]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

channels = sorted(adata_full.obs['Channel'].unique())
print(f"N channels: {len(channels)}")  # expect 63
for ch in channels[1:]:               # 63 dummy columns
    df_cov['channel_%s' % ch] = (
        adata_full.obs['Channel'] == ch).astype(int)

print("Shape:", df_cov.shape)         # expect (266271, 64)

In [ ]:
### saving to csv files
df_cov.to_csv(OUT_PATH, sep="\t")

## Making covariate file for CB_standard_design.h5ad

In [ ]:
DATA_FILE='Data/sc_annot_files/CB_standard_design.h5ad'
OUT_PATH='Data/sc_annot_files_qc/CB_standard_design_covs.tsv'

In [ ]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [ ]:
adata_full

In [ ]:
# Check filtering status
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Predicted doublets:", adata_full.obs['pred_dbl'].sum())

# Check key columns
print("\nDonor value counts:")
print(adata_full.obs['Donor'].value_counts().head(15))

print("\nChannel value counts:")
print(adata_full.obs['Channel'].value_counts().head(15))

print("\nGroup value counts:")
print(adata_full.obs['Group'].value_counts())

print("\nAnno value counts:")
print(adata_full.obs['anno'].value_counts().head(15))

# Check X
print("\nX dtype:", adata_full.X.dtype)
print("X sample values:")
import numpy as np
sample = adata_full.X[:3, :5]
print(sample.toarray() if hasattr(sample, 'toarray') else sample)
print("X min/max:", adata_full.X.min(), adata_full.X.max())
print("Has raw:", adata_full.raw is not None)

In [ ]:
# Check if low-quality cells are already removed
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Any predicted doublets:", adata_full.obs['pred_dbl'].sum())

In [ ]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

channels = sorted(adata_full.obs['Channel'].unique())
print(f"N channels: {len(channels)}")
for ch in channels[1:]:   # drop first as reference
    df_cov['channel_%s' % ch] = (
        adata_full.obs['Channel'] == ch).astype(int)

In [ ]:
# How many cells per channel?
print(adata_full.obs['Channel'].value_counts().describe())

# Do channel names look like sample IDs or run IDs?
print(adata_full.obs['Channel'].value_counts().head(10))

### these are actual donors

In [ ]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['n_genes']

channels = sorted(adata_full.obs['Channel'].unique())
print(f"N channels: {len(channels)}")  # expect 64
for ch in channels[1:]:               # 64 dummy columns
    df_cov['channel_%s' % ch] = (
        adata_full.obs['Channel'] == ch).astype(int)

print("Shape:", df_cov.shape)         # expect (239544, 65)

In [ ]:
### saving to csv files
df_cov.to_csv(OUT_PATH, sep="\t")

## Making covariate file for CB_pooling_and_control.h5ad

In [ ]:
DATA_FILE='Data/sc_annot_files/CB_pooling_and_hashing.h5ad'
OUT_PATH='Data/sc_annot_files_qc/CB_pooling_and_hashing_covs.tsv'

In [ ]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [ ]:
adata_full

In [ ]:
# Quality check
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())

# No pred_dbl column this time — check if it's somewhere else
print("obs columns:", adata_full.obs.columns.tolist())

# Channel structure
print("\nChannel value counts (describe):")
print(adata_full.obs['Channel'].value_counts().describe())
print("\nChannel top 10:")
print(adata_full.obs['Channel'].value_counts().head(10))

# Assignment — donor or demux classification?
print("\nAssignment unique values:", adata_full.obs['assignment'].nunique())
print(adata_full.obs['assignment'].value_counts().head(10))

# X check
sample = adata_full.X[:3, :5]
print("\nX sample:", sample.toarray() if hasattr(sample, 'toarray') else sample)
print("X min/max:", adata_full.X.min(), adata_full.X.max())

### unlike the bone marrow one, this one doesn't have any unlabeled data

In [ ]:
# Check if low-quality cells are already removed
print("N cells:", adata_full.shape[0])
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())

In [ ]:
# Quality check first
print("Shape:", adata_full.shape)
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())

# Check if pred_dbl exists and filter if needed
if 'pred_dbl' in adata_full.obs.columns:
    print("Doublets:", adata_full.obs['pred_dbl'].sum())
    adata = adata_full[adata_full.obs['pred_dbl'] == False].copy()
else:
    print("No pred_dbl column — checking assignment for blanks")
    valid = adata_full.obs['assignment'].isin([
        'MantonCB1', 'MantonCB2', 'MantonCB3',
        'MantonCB4', 'MantonCB5', 'MantonCB6', 'MantonCB7'
    ])
    print("Invalid assignments:", (~valid).sum())
    adata = adata_full[valid].copy()

print("Clean shape:", adata.shape)

# Generate covariate file — donor dummies
df_cov = pd.DataFrame(index=adata.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata.obs['n_genes']

# Also add channel as it captures the two processing batches
# With only 2 channels this is just 1 dummy column
df_cov['channel_CBP'] = (adata.obs['Channel'] == 'CBP').astype(int)

# Donor dummies — drop MantonCB1 as reference
donors = sorted(adata.obs['assignment'].unique())
print("Donors:", donors)
for donor in donors[1:]:   # 6 dummy columns
    df_cov['donor_%s' % donor] = (
        adata.obs['assignment'] == donor).astype(int)

print("\nCov shape:", df_cov.shape)
# Expect: (n_cells, 9) — const + n_genes + 1 channel + 6 donor dummies
print(df_cov.head())

In [ ]:
### saving to csv files
df_cov.to_csv(OUT_PATH, sep="\t")

## Making covariate file for CB_extra.h5ad

In [ ]:
DATA_FILE='Data/sc_annot_files/CB_extra.h5ad'
OUT_PATH='Data/sc_annot_files_qc/CB_extra_covs.tsv'

In [ ]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full.X = adata_full.raw.X

In [ ]:
adata_full

In [ ]:
# QC check
print("Shape:", adata_full.shape)
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())
print("Predicted doublets:", adata_full.obs['pred_dbl'].sum())

# Channel and Donor structure
print("\nChannel value counts (describe):")
print(adata_full.obs['Channel'].value_counts().describe())
print("\nChannel top 10:")
print(adata_full.obs['Channel'].value_counts().head(10))

print("\nDonor value counts:")
print(adata_full.obs['Donor'].value_counts())

In [ ]:
# Remove doublets if any
print("Doublets:", adata_full.obs['pred_dbl'].sum())
adata = adata_full[adata_full.obs['pred_dbl'] == False].copy()
print("Clean shape:", adata.shape)

# Channel dummies
df_cov = pd.DataFrame(index=adata.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata.obs['n_genes']

channels = sorted(adata.obs['Channel'].unique())
print(f"N channels: {len(channels)}")
print(f"Reference (dropped): {channels[0]}")

for ch in channels[1:]:
    df_cov['channel_%s' % ch] = (
        adata.obs['Channel'] == ch).astype(int)

print(f"Cov shape: {df_cov.shape}")
print("Saved.")

In [ ]:
# Quality check first
print("Shape:", adata_full.shape)
print("Min n_genes:", adata_full.obs['n_genes'].min())
print("Max percent_mito:", adata_full.obs['percent_mito'].max())

# Check if pred_dbl exists and filter if needed
if 'pred_dbl' in adata_full.obs.columns:
    print("Doublets:", adata_full.obs['pred_dbl'].sum())
    adata = adata_full[adata_full.obs['pred_dbl'] == False].copy()
else:
    print("No pred_dbl column — checking assignment for blanks")
    valid = adata_full.obs['assignment'].isin([
        'MantonCB1', 'MantonCB2', 'MantonCB3',
        'MantonCB4', 'MantonCB5', 'MantonCB6', 'MantonCB7'
    ])
    print("Invalid assignments:", (~valid).sum())
    adata = adata_full[valid].copy()

print("Clean shape:", adata.shape)

# Generate covariate file — donor dummies
df_cov = pd.DataFrame(index=adata.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata.obs['n_genes']

# Also add channel as it captures the two processing batches
# With only 2 channels this is just 1 dummy column
df_cov['channel_CBP'] = (adata.obs['Channel'] == 'CBP').astype(int)

# Donor dummies — drop MantonCB1 as reference
donors = sorted(adata.obs['assignment'].unique())
print("Donors:", donors)
for donor in donors[1:]:   # 6 dummy columns
    df_cov['donor_%s' % donor] = (
        adata.obs['assignment'] == donor).astype(int)

print("\nCov shape:", df_cov.shape)
# Expect: (n_cells, 9) — const + n_genes + 1 channel + 6 donor dummies
print(df_cov.head())

In [ ]:
print(f"Cov shape: {df_cov.shape}")
df_cov.to_csv(OUT_PATH, sep="\t")

## Making covariate file for onek1k.h5ad

In [5]:
DATA_FILE='Data/sc_annot_files/onek1k_sc.h5ad'
OUT_PATH='Data/sc_annot_files_qc/onek1k_sc_covs.tsv'

In [6]:
adata_full = sc.read_h5ad(DATA_FILE)
adata_full

AnnData object with n_obs × n_vars = 1248980 × 35528
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'donor_id', 'pool_number', 'predicted.celltype.l2', 'predicted.celltype.l2.score', 'age', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'cell_type_ontology_term_id_colors', 'citation', 'default_embedding', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'titl

In [7]:
sample = adata_full.X[:3, :5]
print(sample.toarray() if hasattr(sample, 'toarray') else sample)
print("X min/max:", adata_full.X.min(), adata_full.X.max())
print("X dtype:", adata_full.X.dtype)

# Check key obs columns
print("\ndonor_id unique:", adata_full.obs['donor_id'].nunique())
print(adata_full.obs['donor_id'].value_counts().head(10))

print("\npool_number unique:", adata_full.obs['pool_number'].nunique())
print(adata_full.obs['pool_number'].value_counts().head(10))

print("\nassay:", adata_full.obs['assay'].value_counts())
print("\ntissue:", adata_full.obs['tissue'].value_counts())
print("\nsex:", adata_full.obs['sex'].value_counts())
print("\nage describe:", adata_full.obs['age'].describe())

# QC
print("\nnFeature_RNA min:", adata_full.obs['nFeature_RNA'].min())
print("percent.mt max:", adata_full.obs['percent.mt'].max())

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
X min/max: 0.0 16282.0
X dtype: float32

donor_id unique: 981
donor_id
650_651      3511
764_765      3146
773_774      2731
1008_1009    2684
765_766      2621
700_701      2425
772_773      2417
361_362      2398
781_782      2256
832_833      2216
Name: count, dtype: int64

pool_number unique: 75
pool_number
20    20011
13    19751
39    19711
3     19103
19    18987
54    18979
41    18940
10    18892
11    18853
25    18592
Name: count, dtype: int64

assay: assay
10x 3' v2    1248980
Name: count, dtype: int64

tissue: tissue
blood    1248980
Name: count, dtype: int64

sex: sex
female    731202
male      517778
Name: count, dtype: int64

age describe: count    1.248980e+06
mean     6.350877e+01
std      1.661744e+01
min      1.900000e+01
25%      5.600000e+01
50%      6.700000e+01
75%      7.500000e+01
max      9.700000e+01
Name: age, dtype: float64

nFeature_RNA min: 232
percent.mt max: 7.832512315270936


In [8]:
### the minimum cell is 232, to keep consistent with HCA I will filter them
print("Before filter:", adata_full.shape)
print("Cells below 500 genes:", (adata_full.obs['nFeature_RNA'] < 500).sum())

# Filter to match other datasets
sc.pp.filter_cells(adata_full, min_genes=500)
print("After filter:", adata_full.shape)

# Also filter genes to match Martin's min_cells=50
sc.pp.filter_genes(adata_full, min_cells=50)
print("After gene filter:", adata_full.shape)

Before filter: (1248980, 35528)
Cells below 500 genes: 73396
After filter: (1174789, 35528)
After gene filter: (1174789, 22459)


In [9]:
df_cov = pd.DataFrame(index=adata_full.obs.index)
df_cov['const']   = 1
df_cov['n_genes'] = adata_full.obs['nFeature_RNA']

pools = sorted(adata_full.obs['pool_number'].unique())
for pool in pools[1:]:
    df_cov['pool_%s' % pool] = (
        adata_full.obs['pool_number'] == pool).astype(int)

In [10]:
df_cov.to_csv(OUT_PATH, sep="\t")

# Save filtered h5ad — must match the .cov index
adata_full.write("Data/sc_annot_files/onek1k_sc_filtered.h5ad")
print("Saved onek1k.cov and onek1k_filtered.h5ad")

Saved onek1k.cov and onek1k_filtered.h5ad
